# Nine of the ten reports carrying `Onychomycosis` are one patient

The ranking pointed at nail fungus. This notebook follows the pairs
back to the documents that produced them.

In [1]:
import sys

sys.path.insert(0, "../src")

import duckdb
import pandas

from hindsight.analysis.crowding import breadth, overlap, wide_reports

PARTITION = "../data/parquet/year=2025/quarter=1/part=0001-of-0028"

In [2]:
carriers = duckdb.sql(f'''
    SELECT r.safetyreportid, count(DISTINCT d.medicinalproduct) AS drugs
    FROM '{PARTITION}/report_reaction.parquet' AS r
    JOIN '{PARTITION}/report_drug.parquet' AS d USING (safetyreportid)
    WHERE r.reactionmeddrapt = 'Onychomycosis'
    GROUP BY 1
    ORDER BY 2 DESC
''').df()

carriers

,safetyreportid,drugs
0,25021632,96
1,25044641,91
2,23733354,89
3,24757080,84
4,24965286,78
5,24361485,77
6,24087637,73
7,24816316,73
8,24388883,66
9,20836166,3


Ten reports carry the term. Nine of them name between 66 and 96
distinct drugs; the tenth names three and is an ordinary report.

A report naming 90 drugs and 10 events asserts 900 drug–event pairs.
Nine such reports put `a = 9` on every one of them.

In [3]:
cluster = carriers[carriers.drugs > 10].safetyreportid.tolist()

scores = pandas.DataFrame(
    overlap(cluster, root="../data/parquet"),
    columns=["report", "other", "jaccard"],
)

scores.jaccard.describe()[["min", "50%", "max"]]

min    0.376238
50%    0.480762
max    0.908163
Name: jaccard, dtype: float64

## Jaccard 0.38 minimum, 0.48 median, 0.91 maximum

These are not nine patients who happen to be on long medication
lists. Two of them share a `companynumb` and an identical
ten-term reaction list — the same case, filed twice.

In [4]:
duckdb.sql(f'''
    SELECT safetyreportid, companynumb, occurcountry, serious, receiptdate
    FROM '{PARTITION}/report.parquet'
    WHERE safetyreportid IN {tuple(cluster)}
    ORDER BY companynumb
''')

┌────────────────┬───────────────────────────────────────────┬──────────────┬─────────┬─────────────┐
│ safetyreportid │                companynumb                │ occurcountry │ serious │ receiptdate │
│    varchar     │                  varchar                  │   varchar    │ varchar │   varchar   │
├────────────────┼───────────────────────────────────────────┼──────────────┼─────────┼─────────────┤
│ 24757080       │ CA-BEH-2024187690                         │ CA           │ 1       │ 20250311    │
│ 24965286       │ CA-BIOCON BIOLOGICS LIMITED-BBL2025000590 │ CA           │ 1       │ 20250213    │
│ 24087637       │ CA-JNJFOC-20240708638                     │ CA           │ 1       │ 20250319    │
│ 24361485       │ CA-JNJFOC-20240931166                     │ CA           │ 1       │ 20250217    │
│ 23733354       │ CA-PURDUE-USA-2024-0308932                │ CA           │ 1       │ 20250318    │
│ 24388883       │ CA-PURDUE-USA-2024-0312263                │ CA           │ 1   

All Canadian, all serious, filed between January and March 2025 by
six different manufacturers — every company whose product was on the
list reported it independently.

Two fixes look obvious and neither works.

In [5]:
duckdb.sql(f'''
    SELECT drugcharacterization, count(*) AS rows
    FROM '{PARTITION}/report_drug.parquet'
    WHERE safetyreportid IN {tuple(cluster)}
    GROUP BY 1
''')

┌──────────────────────┬───────┐
│ drugcharacterization │ rows  │
│       varchar        │ int64 │
├──────────────────────┼───────┤
│ 1                    │  2345 │
│ 2                    │    29 │
└──────────────────────┴───────┘

Disproportionality is conventionally run on suspect drugs only.
Here **every** drug in these reports is marked suspect, so
`drugcharacterization` removes nothing. The screening criterion
removes nothing either — notebook 02 measured that.

What does separate them is the shape of the document.

In [6]:
breadth(root="../data/parquet")

{'cut': 27.0, 'median': 2.0, 'widest': 121, 'reports': 12000}

## The median report names two drugs; the 99th percentile names 27

The cluster names 66 to 96. That gap is what makes a quantile a
usable cut rather than a hopeful one — nothing sits on the line
arguing about which side it belongs on.

A constant would not travel: this is one partition of one export,
and the corpus spans 2004 to 2025.

In [7]:
wide = wide_reports(cut=27, root="../data/parquet")

len(wide), len(wide) / 12000

(125, 0.010416666666666666)

125 reports of 12,000 — 1.04%. What that 1% does to the pair table
is the M0 result, and it is in `reports/m0.qmd`.

**This is not a deduplication rule.** Deciding that two reports
describe one case needs entity resolution, which is M2. A high
crowding score says the evidence cannot tell a repeated case from a
real one — not that the pair is false. A patient on 90 drugs who has
an adverse reaction is a real patient.